In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.stats import poisson

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
BASE_DIR = Path("..")

RAW_PATH = BASE_DIR / "data" / "processed" / "wc_matches_1990_2022.csv"
FEATURE_PATH = BASE_DIR / "data" / "processed" / "match_features_wc_1990_2022.csv"

matches = pd.read_csv(RAW_PATH)
matches["match_date"] = pd.to_datetime(matches["match_date"])

matches = matches.sort_values("match_date").reset_index(drop=True)
matches.head()

,match_date,year,stage,team_a,team_b,goals_a,goals_b,xg_a,xg_b,host_country,team_a_win,team_b_win,draw
0,1990-06-08,1990,Group stage,Argentina,Cameroon,0,1,NaN,NaN,Italy,False,True,False
1,1990-06-09,1990,Group stage,United Arab Emirates,Colombia,0,2,NaN,NaN,Italy,False,True,False
2,1990-06-09,1990,Group stage,Italy,Austria,1,0,NaN,NaN,Italy,True,False,False
3,1990-06-09,1990,Group stage,Soviet Union,Romania,0,2,NaN,NaN,Italy,False,True,False
4,1990-06-10,1990,Group stage,United States,Czechoslovakia,1,5,NaN,NaN,Italy,False,True,False


In [3]:
matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   match_date    552 non-null    datetime64[us]
 1   year          552 non-null    int64         
 2   stage         552 non-null    str           
 3   team_a        552 non-null    str           
 4   team_b        552 non-null    str           
 5   goals_a       552 non-null    int64         
 6   goals_b       552 non-null    int64         
 7   xg_a          128 non-null    float64       
 8   xg_b          128 non-null    float64       
 9   host_country  552 non-null    str           
 10  team_a_win    552 non-null    bool          
 11  team_b_win    552 non-null    bool          
 12  draw          552 non-null    bool          
dtypes: bool(3), datetime64[us](1), float64(2), int64(3), str(4)
memory usage: 64.7 KB


In [4]:
matches[["team_a", "team_b", "goals_a", "goals_b", "xg_a", "xg_b"]].head()

,team_a,team_b,goals_a,goals_b,xg_a,xg_b
0,Argentina,Cameroon,0,1,NaN,NaN
1,United Arab Emirates,Colombia,0,2,NaN,NaN
2,Italy,Austria,1,0,NaN,NaN
3,Soviet Union,Romania,0,2,NaN,NaN
4,United States,Czechoslovakia,1,5,NaN,NaN


In [5]:
matches = matches.dropna(subset=["goals_a", "goals_b", "xg_a", "xg_b"]).copy()

matches["goals_a"] = matches["goals_a"].astype(int)
matches["goals_b"] = matches["goals_b"].astype(int)

matches.shape

(128, 13)

In [6]:
DEFAULT_STATE = {
    "gf_avg": 1.30,
    "ga_avg": 1.30,
    "xgf_avg": 1.30,
    "xga_avg": 1.30,
    "points_avg": 1.00,
    "matches_played": 0
}

In [7]:
def get_state(team, history, window=5):
    games = history.get(team, [])

    if not games:
        return DEFAULT_STATE.copy()

    recent = games[-window:]

    return {
        "gf_avg": np.mean([g["gf"] for g in recent]),
        "ga_avg": np.mean([g["ga"] for g in recent]),
        "xgf_avg": np.mean([g["xgf"] for g in recent]),
        "xga_avg": np.mean([g["xga"] for g in recent]),
        "points_avg": np.mean([g["points"] for g in recent]),
        "matches_played": len(games)
    }

In [8]:
def match_points(goals_for, goals_against):
    if goals_for > goals_against:
        return 3
    if goals_for == goals_against:
        return 1
    return 0

In [9]:
history = {}
rows = []

for _, match in matches.iterrows():
    team_a = match["team_a"]
    team_b = match["team_b"]

    a = get_state(team_a, history)
    b = get_state(team_b, history)

    rows.append({
        "match_date": match["match_date"],
        "year": match["year"],
        "tournament": "FIFA World Cup",
        "stage": match["stage"],

        "team_a": team_a,
        "team_b": team_b,

        "gf_avg_a": a["gf_avg"],
        "ga_avg_a": a["ga_avg"],
        "xgf_avg_a": a["xgf_avg"],
        "xga_avg_a": a["xga_avg"],
        "points_avg_a": a["points_avg"],
        "matches_played_a": a["matches_played"],

        "gf_avg_b": b["gf_avg"],
        "ga_avg_b": b["ga_avg"],
        "xgf_avg_b": b["xgf_avg"],
        "xga_avg_b": b["xga_avg"],
        "points_avg_b": b["points_avg"],
        "matches_played_b": b["matches_played"],

        "gf_diff": a["gf_avg"] - b["gf_avg"],
        "ga_diff": b["ga_avg"] - a["ga_avg"],
        "xgf_diff": a["xgf_avg"] - b["xgf_avg"],
        "xga_diff": b["xga_avg"] - a["xga_avg"],
        "form_diff": a["points_avg"] - b["points_avg"],
        "experience_diff": a["matches_played"] - b["matches_played"],

        "goals_a": match["goals_a"],
        "goals_b": match["goals_b"],
        "xg_a": match["xg_a"],
        "xg_b": match["xg_b"]
    })

    history.setdefault(team_a, []).append({
        "gf": match["goals_a"],
        "ga": match["goals_b"],
        "xgf": match["xg_a"],
        "xga": match["xg_b"],
        "points": match_points(match["goals_a"], match["goals_b"])
    })

    history.setdefault(team_b, []).append({
        "gf": match["goals_b"],
        "ga": match["goals_a"],
        "xgf": match["xg_b"],
        "xga": match["xg_a"],
        "points": match_points(match["goals_b"], match["goals_a"])
    })

match_features = pd.DataFrame(rows)
match_features.head()

,match_date,year,tournament,stage,team_a,team_b,gf_avg_a,ga_avg_a,xgf_avg_a,xga_avg_a,...,gf_diff,ga_diff,xgf_diff,xga_diff,form_diff,experience_diff,goals_a,goals_b,xg_a,xg_b
0,2018-06-14,2018,FIFA World Cup,Group stage,Russia,Saudi Arabia,1.3,1.3,1.3,1.3,...,0.0,0.0,0.0,0.0,0.0,0,5,0,1.7,0.2
1,2018-06-15,2018,FIFA World Cup,Group stage,Portugal,Spain,1.3,1.3,1.3,1.3,...,0.0,0.0,0.0,0.0,0.0,0,3,3,1.2,1.6
2,2018-06-15,2018,FIFA World Cup,Group stage,Morocco,IR Iran,1.3,1.3,1.3,1.3,...,0.0,0.0,0.0,0.0,0.0,0,0,1,0.7,1.0
3,2018-06-15,2018,FIFA World Cup,Group stage,Egypt,Uruguay,1.3,1.3,1.3,1.3,...,0.0,0.0,0.0,0.0,0.0,0,0,1,0.3,1.6
4,2018-06-16,2018,FIFA World Cup,Group stage,Croatia,Nigeria,1.3,1.3,1.3,1.3,...,0.0,0.0,0.0,0.0,0.0,0,2,0,1.5,0.7


In [10]:
match_features.to_csv(FEATURE_PATH, index=False)

print("saved:", FEATURE_PATH)
print(match_features.shape)

saved: ..\data\processed\match_features_wc_1990_2022.csv
(128, 28)


In [11]:
feature_cols = [
    "gf_diff",
    "ga_diff",
    "xgf_diff",
    "xga_diff",
    "form_diff",
    "experience_diff",
    "gf_avg_a",
    "ga_avg_a",
    "xgf_avg_a",
    "xga_avg_a",
    "points_avg_a",
    "gf_avg_b",
    "ga_avg_b",
    "xgf_avg_b",
    "xga_avg_b",
    "points_avg_b"
]

target_cols = ["xg_a", "xg_b"]

In [12]:
train = match_features[match_features["year"] < 2022].copy()
test = match_features[match_features["year"] == 2022].copy()

X_train = train[feature_cols]
y_train = train[target_cols]

X_test = test[feature_cols]
y_test = test[target_cols]

X_train.shape, X_test.shape

((64, 16), (64, 16))

In [13]:
model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=4,
        random_state=42
    )
)

model.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,RandomForestR...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[RandomForestR...ndom_state=42), RandomForestR...ndom_state=42)]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['gf_diff','ga_diff','xgf_diff',...,'xgf_avg_b','xga_avg_b','points_avg_b']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,16
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'


In [14]:
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2  :", round(r2, 4))

MAE : 0.6626
RMSE: 0.886
R2  : 0.0662


In [15]:
pred_df = test[["match_date", "team_a", "team_b", "xg_a", "xg_b", "goals_a", "goals_b"]].copy()

pred_df["pred_xg_a"] = pred[:, 0]
pred_df["pred_xg_b"] = pred[:, 1]

pred_df.head(10)

,match_date,team_a,team_b,xg_a,xg_b,goals_a,goals_b,pred_xg_a,pred_xg_b
64,2022-11-20,Qatar,Ecuador,0.3,1.2,0,2,1.376620,1.097819
65,2022-11-21,United States,Wales,0.8,1.5,1,1,1.376620,1.097819
66,2022-11-21,Senegal,Netherlands,0.9,0.7,0,2,1.235976,1.292211
67,2022-11-21,England,IR Iran,2.1,1.4,6,2,1.249528,1.278214
68,2022-11-22,France,Australia,4.0,0.5,4,1,1.391836,0.845565
69,2022-11-22,Mexico,Poland,0.7,0.9,0,0,1.362813,1.164627
70,2022-11-22,Denmark,Tunisia,1.4,0.9,0,0,1.009331,1.504893
71,2022-11-22,Argentina,Saudi Arabia,2.2,0.1,1,2,1.400103,1.135673
72,2022-11-23,Belgium,Canada,0.8,2.4,1,0,1.278099,1.150499
73,2022-11-23,Spain,Costa Rica,3.5,0.0,7,0,2.196613,0.963249


In [16]:
def score_matrix(lambda_a, lambda_b, max_goals=6):
    matrix = np.zeros((max_goals + 1, max_goals + 1))

    for goals_a in range(max_goals + 1):
        for goals_b in range(max_goals + 1):
            matrix[goals_a, goals_b] = poisson.pmf(goals_a, lambda_a) * poisson.pmf(goals_b, lambda_b)

    return matrix

In [17]:
def match_probabilities(lambda_a, lambda_b, max_goals=6):
    matrix = score_matrix(lambda_a, lambda_b, max_goals)

    team_a_win = np.tril(matrix, -1).sum()
    draw = np.trace(matrix)
    team_b_win = np.triu(matrix, 1).sum()

    most_likely = np.unravel_index(np.argmax(matrix), matrix.shape)

    return {
        "team_a_win": team_a_win,
        "draw": draw,
        "team_b_win": team_b_win,
        "most_likely_score": most_likely,
        "score_matrix": matrix
    }

In [18]:
sample = pred_df.iloc[0]

result = match_probabilities(sample["pred_xg_a"], sample["pred_xg_b"])

print(sample["team_a"], "vs", sample["team_b"])
print("Pred xG:", round(sample["pred_xg_a"], 2), "-", round(sample["pred_xg_b"], 2))
print("Team A win:", round(result["team_a_win"] * 100, 2), "%")
print("Draw      :", round(result["draw"] * 100, 2), "%")
print("Team B win:", round(result["team_b_win"] * 100, 2), "%")
print("Most likely score:", result["most_likely_score"])

Qatar vs Ecuador
Pred xG: 1.38 - 1.1
Team A win: 43.17 %
Draw      : 26.84 %
Team B win: 29.91 %
Most likely score: (np.int64(1), np.int64(1))


In [19]:
def latest_team_state(team):
    a_rows = match_features[match_features["team_a"] == team]
    b_rows = match_features[match_features["team_b"] == team]

    records = []

    for _, r in a_rows.iterrows():
        records.append({
            "date": r["match_date"],
            "gf_avg": r["gf_avg_a"],
            "ga_avg": r["ga_avg_a"],
            "xgf_avg": r["xgf_avg_a"],
            "xga_avg": r["xga_avg_a"],
            "points_avg": r["points_avg_a"],
            "matches_played": r["matches_played_a"]
        })

    for _, r in b_rows.iterrows():
        records.append({
            "date": r["match_date"],
            "gf_avg": r["gf_avg_b"],
            "ga_avg": r["ga_avg_b"],
            "xgf_avg": r["xgf_avg_b"],
            "xga_avg": r["xga_avg_b"],
            "points_avg": r["points_avg_b"],
            "matches_played": r["matches_played_b"]
        })

    if not records:
        return DEFAULT_STATE.copy()

    return pd.DataFrame(records).sort_values("date").iloc[-1].drop("date").to_dict()

In [20]:
def build_prediction_row(team_a, team_b):
    a = latest_team_state(team_a)
    b = latest_team_state(team_b)

    return pd.DataFrame([{
        "gf_diff": a["gf_avg"] - b["gf_avg"],
        "ga_diff": b["ga_avg"] - a["ga_avg"],
        "xgf_diff": a["xgf_avg"] - b["xgf_avg"],
        "xga_diff": b["xga_avg"] - a["xga_avg"],
        "form_diff": a["points_avg"] - b["points_avg"],
        "experience_diff": a["matches_played"] - b["matches_played"],

        "gf_avg_a": a["gf_avg"],
        "ga_avg_a": a["ga_avg"],
        "xgf_avg_a": a["xgf_avg"],
        "xga_avg_a": a["xga_avg"],
        "points_avg_a": a["points_avg"],

        "gf_avg_b": b["gf_avg"],
        "ga_avg_b": b["ga_avg"],
        "xgf_avg_b": b["xgf_avg"],
        "xga_avg_b": b["xga_avg"],
        "points_avg_b": b["points_avg"]
    }])

In [21]:
def predict_match(team_a, team_b):
    if team_a == team_b:
        raise ValueError("Choose two different teams.")

    X = build_prediction_row(team_a, team_b)
    xg = model.predict(X[feature_cols])[0]
    xg_a = max(0.05, xg[0])
    xg_b = max(0.05, xg[1])

    probs = match_probabilities(xg_a, xg_b)

    return {
        "team_a": team_a,
        "team_b": team_b,
        "xg_a": xg_a,
        "xg_b": xg_b,
        "team_a_win": probs["team_a_win"],
        "draw": probs["draw"],
        "team_b_win": probs["team_b_win"],
        "most_likely_score": (
    int(probs["most_likely_score"][0]),
    int(probs["most_likely_score"][1])
)
    }

In [22]:
result = predict_match("Argentina", "England")

print(f'{result["team_a"]} vs {result["team_b"]}')
print("xG:", round(result["xg_a"], 2), "-", round(result["xg_b"], 2))
print(f'{result["team_a"]} win:', round(result["team_a_win"] * 100, 2), "%")
print("Draw:", round(result["draw"] * 100, 2), "%")
print(f'{result["team_b"]} win:', round(result["team_b_win"] * 100, 2), "%")
print("Most likely score:", result["most_likely_score"])

Argentina vs England
xG: 1.42 - 0.67
Argentina win: 55.14 %
Draw: 27.28 %
England win: 17.51 %
Most likely score: (1, 0)


In [23]:
teams = sorted(set(match_features["team_a"]).union(set(match_features["team_b"])))
teams[:20], len(teams)

(['Argentina',
  'Australia',
  'Belgium',
  'Brazil',
  'Cameroon',
  'Canada',
  'Colombia',
  'Costa Rica',
  'Croatia',
  'Denmark',
  'Ecuador',
  'Egypt',
  'England',
  'France',
  'Germany',
  'Ghana',
  'IR Iran',
  'Iceland',
  'Japan',
  'Korea Republic'],
 40)

In [24]:
import joblib

MODEL_PATH = BASE_DIR / "models" / "xg_random_forest_wc_baseline.pkl"
MODEL_PATH.parent.mkdir(exist_ok=True)

joblib.dump(model, MODEL_PATH)

print("saved:", MODEL_PATH)

saved: ..\models\xg_random_forest_wc_baseline.pkl


In [25]:
RATING_PATH = BASE_DIR / "data" / "processed" / "team_ratings_2026.csv"

ratings = pd.read_csv(RATING_PATH)
ratings.head()

,team,confederation,fifa_rank,host,group,rank_strength,host_boost,confed_bonus,oracle_rating,attack_index,defense_index,momentum_index
0,Mexico,CONCACAF,15,1,A,86.0,3.0,1.0,77.10,79.2,77.28,83.78
1,Czech Republic,UEFA,41,0,A,60.0,0.0,3.0,54.00,55.8,54.90,58.50
2,South Korea,AFC,25,0,A,76.0,0.0,1.0,65.60,69.0,67.58,73.40
3,South Africa,CAF,60,0,A,41.0,0.0,1.5,36.35,37.8,37.13,39.84
4,Canada,CONCACAF,30,1,B,71.0,3.0,1.0,64.35,65.7,64.08,69.34


In [26]:
ratings[["team", "oracle_rating", "attack_index", "defense_index", "momentum_index", "host"]].head()

,team,oracle_rating,attack_index,defense_index,momentum_index,host
0,Mexico,77.10,79.2,77.28,83.78,1
1,Czech Republic,54.00,55.8,54.90,58.50,0
2,South Korea,65.60,69.0,67.58,73.40,0
3,South Africa,36.35,37.8,37.13,39.84,0
4,Canada,64.35,65.7,64.08,69.34,1


In [27]:
rating_map = ratings.set_index("team").to_dict(orient="index")

def get_rating(team):
    if team not in rating_map:
        raise ValueError(f"{team} not found in rating table")
    return rating_map[team]

In [28]:
def rating_features(team_a, team_b):
    a = get_rating(team_a)
    b = get_rating(team_b)

    return {
        "oracle_diff": a["oracle_rating"] - b["oracle_rating"],
        "attack_diff": a["attack_index"] - b["attack_index"],
        "defense_diff": a["defense_index"] - b["defense_index"],
        "momentum_diff": a["momentum_index"] - b["momentum_index"],
        "team_a_host": a["host"],
        "team_b_host": b["host"],
        "team_a_oracle": a["oracle_rating"],
        "team_b_oracle": b["oracle_rating"],
        "team_a_attack": a["attack_index"],
        "team_b_attack": b["attack_index"],
        "team_a_defense": a["defense_index"],
        "team_b_defense": b["defense_index"],
        "team_a_momentum": a["momentum_index"],
        "team_b_momentum": b["momentum_index"]
    }

In [29]:
rating_features("Argentina", "England")

{'oracle_diff': 0.8499999999999943,
 'attack_diff': 0.9000000000000057,
 'defense_diff': 0.8800000000000097,
 'momentum_diff': 0.9699999999999989,
 'team_a_host': 0,
 'team_b_host': 0,
 'team_a_oracle': 86.3,
 'team_b_oracle': 85.45,
 'team_a_attack': 90.0,
 'team_b_attack': 89.1,
 'team_a_defense': 88.34,
 'team_b_defense': 87.46,
 'team_a_momentum': 95.08,
 'team_b_momentum': 94.11}

In [30]:
def adjust_xg(base_xg_a, base_xg_b, team_a, team_b):
    f = rating_features(team_a, team_b)

    rating_boost = 0.012 * f["oracle_diff"]
    attack_boost = 0.010 * f["attack_diff"]
    defense_boost = 0.008 * f["defense_diff"]
    momentum_boost = 0.006 * f["momentum_diff"]
    host_boost = 0.12 * (f["team_a_host"] - f["team_b_host"])

    net_boost = rating_boost + attack_boost + defense_boost + momentum_boost + host_boost

    adj_xg_a = base_xg_a + net_boost
    adj_xg_b = base_xg_b - net_boost

    adj_xg_a = max(0.15, min(adj_xg_a, 3.50))
    adj_xg_b = max(0.15, min(adj_xg_b, 3.50))

    return adj_xg_a, adj_xg_b

In [31]:
def predict_match_hybrid(team_a, team_b):
    if team_a == team_b:
        raise ValueError("Choose two different teams.")

    X = build_prediction_row(team_a, team_b)
    base_xg = model.predict(X[feature_cols])[0]

    base_xg_a = max(0.05, base_xg[0])
    base_xg_b = max(0.05, base_xg[1])

    final_xg_a, final_xg_b = adjust_xg(base_xg_a, base_xg_b, team_a, team_b)

    probs = match_probabilities(final_xg_a, final_xg_b)

    return {
        "team_a": team_a,
        "team_b": team_b,
        "base_xg_a": base_xg_a,
        "base_xg_b": base_xg_b,
        "xg_a": final_xg_a,
        "xg_b": final_xg_b,
        "team_a_win": probs["team_a_win"],
        "draw": probs["draw"],
        "team_b_win": probs["team_b_win"],
        "most_likely_score": (
            int(probs["most_likely_score"][0]),
            int(probs["most_likely_score"][1])
        )
    }

In [32]:
result = predict_match_hybrid("Argentina", "England")

print(f'{result["team_a"]} vs {result["team_b"]}')
print("Base xG :", round(result["base_xg_a"], 2), "-", round(result["base_xg_b"], 2))
print("Final xG:", round(result["xg_a"], 2), "-", round(result["xg_b"], 2))
print(f'{result["team_a"]} win:', round(result["team_a_win"] * 100, 2), "%")
print("Draw:", round(result["draw"] * 100, 2), "%")
print(f'{result["team_b"]} win:', round(result["team_b_win"] * 100, 2), "%")
print("Most likely score:", result["most_likely_score"])

Argentina vs England
Base xG : 1.42 - 0.67
Final xG: 1.45 - 0.63
Argentina win: 56.88 %
Draw: 26.79 %
England win: 16.25 %
Most likely score: (1, 0)


In [33]:
def explain_prediction(team_a, team_b):
    f = rating_features(team_a, team_b)

    explanation = pd.DataFrame({
        "factor": [
            "Oracle rating",
            "Attack strength",
            "Defensive stability",
            "Momentum",
            "Host advantage"
        ],
        "difference": [
            f["oracle_diff"],
            f["attack_diff"],
            f["defense_diff"],
            f["momentum_diff"],
            f["team_a_host"] - f["team_b_host"]
        ]
    })

    return explanation

In [34]:
explain_prediction("Argentina", "England")

,factor,difference
0,Oracle rating,0.85
1,Attack strength,0.90
2,Defensive stability,0.88
3,Momentum,0.97
4,Host advantage,0.00
